In [1]:
import time
import requests
from typing import List, Dict, Any
import os
import re
from pathlib import Path
import wikipediaapi
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility









#### Konfiguration

In [2]:
USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project; contact: s0579120@htw-student.de) Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

URL_PATH_WIKI = Path('../data/wiki')


# Embedding-Modell
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Milvus-Verbindung
connections.connect("default", host="127.0.0.1", port="19530")


#### 1. Quelle: Wikipedia

In [3]:
# 1. Wikipedia als Quelle

wiki = wikipediaapi.Wikipedia(language='de',
                             extract_format=wikipediaapi.ExtractFormat.WIKI,
                             user_agent=USER_AGENT)

# URL's laden
def load_urls(path: Path):
    urls = []
    for file in path.glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for entry in data:
                urls.append(entry["url"])
    return urls
urls_wiki = load_urls(URL_PATH_WIKI)
# urls_wiki

# Wiki texte von unwichtige Teile bereinigen
def remove_references(text: str) -> str:
    stop_sections = ['Einzelnachweise', 'Weblinks', 'Literatur', 'Quellen', 'Fußnoten']
    for section in stop_sections:
        if section in text:
            text = text.split(section)[0]
    return text.strip()

def clean_wiki_text(text):
    text = remove_references(text)
     # Zeilenende normalisieren
    text = text.replace("\r\n", "\n")

    return text.strip()


def load_wikipedia_text(title: str) -> dict:
    page = wiki.page(title)
    if not page.exists():
        return {'error': 'Page not found'}

    text = clean_wiki_text(page.text)
    return {
        'text': text,
        'title': page.title,
        'source': page.fullurl,
        'license': 'CC BY-SA 4.0'
    }


#Titel aus URL extrahieren
def extract_title_from_url(url: str) -> str:
    match = re.search(r'/wiki/([^#]+)', url)
    if match:
        return match.group(1)
    return None


wiki_data = {}
for url in urls_wiki:
    title = extract_title_from_url(url)
    wiki_data[title] = load_wikipedia_text(title)


def get_summary_from_wiki(text: str, n=2) -> str:
    paragraphs = text.split('\n\n')
    summary = ' '.join(paragraphs[:n])
    return summary

def chunk_wiki_text(text: str, max_words: int = 200, overlap: int = 50):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap  # Überlappung für Kontext
        if start < 0:
            start = 0

    return chunks

#Chunking mit metadaten
wiki_chunks = []
for title, data in wiki_data.items():
    if 'text' in data:
        chunks = chunk_wiki_text(data['text'], max_words=200, overlap=50)
        for i, chunk in enumerate(chunks):
            wiki_chunks.append({
                "text": chunk,
                "title": data['title'],
                "url": data['source'],
                "license": data['license']
            })
print(f"{len(wiki_chunks)} Text-Chunks mit Metadaten erstellt.")


# Collection-Schema definieren
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="url", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="license", dtype=DataType.VARCHAR, max_length=64),
]
schema = CollectionSchema(fields, description="Wikipedia Chunks für semantische Suche")

# Collection erstellen
collection_name = "wiki_collection"
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
collection = Collection(name=collection_name, schema=schema)

#Embeddings für Wiki-Chunks erstellen

#model1 = "sentence-transformers/all-MiniLM-L6-v2"
# model2 = "paraphrase-multilingual-MiniLM-L12-v2"
# embedding_model = SentenceTransformer(model2)
wiki_embeddings = embedding_model.encode([chunk['text'] for chunk in wiki_chunks]).astype(np.float32).tolist()
# print(f"Embeddings für {len(wiki_embeddings)} Wiki-Chunks erstellt.")

#Daten in Milvus einfügen
title = [chunk['title'] for chunk in wiki_chunks]
url = [chunk['url'] for chunk in wiki_chunks]
license = [chunk['license'] for chunk in wiki_chunks]
texts = [chunk['text'] for chunk in wiki_chunks]
collection.insert([wiki_embeddings, texts, title, url, license])

#Index erstellen (HNSW)
index_params = {
    "index_type": "HNSW",
    "metric_type": "COSINE",
    "params": {"M": 16, "efConstruction": 200}
}
collection.create_index(field_name="embedding", index_params=index_params)
print(f"Index für Collection '{collection_name}' erstellt.")

collection.load()  #LOAD COLLEKTION MUSS AM ENDE STEHEN!!!!!

#Semantische Suche in Milvus
def semantic_search_milvus_wiki(query: str, k: int = 5):
    q_emb = embedding_model.encode([query]).astype(np.float32).tolist()
    results = collection.search(
        data=q_emb,
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"ef": 50}},
        limit=k,
        output_fields=["text", "title", "url", "license"]
    )
    output = []
    for result in results:
        for hit in result:
            output.append({
                "score": hit.score,
                "text": hit.entity.get("text"),
                "title": hit.entity.get("title"),
                "url": hit.entity.get("url"),
                "license": hit.entity.get("license")
            })
    return output

# Test
# query = "Geschichte, Spreewald"
# results = semantic_search_milvus_wiki(query, k=5)
#
# for r in results:
#     print(f"\n--- Treffer: {r['title']} ---")
#     print(f"Score: {r['score']:.4f}")
#     print(f"URL: {r['url']}")
#     print(f"Text: {r['text'][:500]}...")  # Nur die ersten 300 Zeichen anzeigen


208 Text-Chunks mit Metadaten erstellt.
Index für Collection 'wiki_collection' erstellt.


In [4]:
# 2. eigene JSON-Datei mit data

def load_data(path='../data/ausflugziele'):
    data = []
    for filename in os.listdir(path):
        # Nur JSON-Dateien verarbeiten
        if filename.endswith('.json'):
            # Öffnen und Laden der JSON-Datei im Lesemodus
            with open(os.path.join(path, filename), 'r', encoding='utf-8') as f:
                # Inhalt der JSON-Datei laden
                content = json.load(f)
                if isinstance(content, list):
                    data.extend(content)
                else:
                    data.append(content)
    return data

data = load_data()
# print(f'{len(data)} documents loaded.')

def clean_text(text):
     # Kleinbuchstaben, Entfernen von Sonderzeichen
    text = text.lower().replace(",", " ").replace("!", " ").replace("?", " ").replace("-", " ").replace(")", " ").strip()
    text = text.split()
    return text

docsListe = []
for i, doc in enumerate(data, start=1):
        if 'beschreibung' in doc:
            text = f"""
            Name: {doc.get('name')}
            Ort: {doc.get('ort')}
            Region: {doc.get('region')}
            Öffnungszeiten: {doc.get('oeffnungszeiten')}
            Eintrittspreise: {doc.get('eintrittspreise')}
            Zielgruppe: {doc.get('zielgruppe')}
            Kategorie: {doc.get('kategorie')}
            Beschreibung: {doc.get('beschreibung')}
            """
            doc_entry = {
                "name": doc.get("name"),
                "ort": doc.get("ort"),
                "region": doc.get("region"),
                "oeffnungszeiten": doc.get("oeffnungszeiten"),
                "eintrittspreise": doc.get("eintrittspreise"),
                "zielgruppe": doc.get("zielgruppe"),
                "kategorie": doc.get("kategorie"),
                "text_tokens": clean_text(doc['beschreibung']),  # tokenisierte Beschreibung
                "text": text.strip()
            }
            docsListe.append(doc_entry)

            # Vorschau ausgeben
            print(f"\n--- Dokument {i}: {doc.get('name', 'Unbekannt')} ---")
            print(f"\n{doc_entry['text'][:800]}...")  # nur die ersten 300 Zeichen

        else:
            print(f"Fehlende Beschreibung in Dokument: {doc.get('name', 'unknown')}")
print(f"{len(docsListe)} Dokumente für RAG vorbereitet.")

#embedding
#model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
#model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
texts = [doc['text'] for doc in docsListe]


embeddings_json = embedding_model.encode(texts)
embeddings = np.array(embeddings_json).astype('float32')

#Daten in Milvus einfügen


collection_name = "ausflug_collection"

if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="name", dtype=DataType.VARCHAR, max_length=512),
    FieldSchema(name="ort", dtype=DataType.VARCHAR, max_length=256),
    FieldSchema(name="region", dtype=DataType.VARCHAR, max_length=256)
]

schema = CollectionSchema(fields, description="Ausflugziele JSON Datenbank")

collection = Collection(collection_name, schema)

texts = [doc["text"] or "" for doc in docsListe]
names = [doc["name"] or "" for doc in docsListe]
orte = [doc["ort"] or "" for doc in docsListe]
regions = [doc["region"] or "" for doc in docsListe]

collection.insert([
    embeddings.tolist(),
    texts,
    names,
    orte,
    regions
])

#Index erstellen
index_params = {

    "index_type": "HNSW",
    "metric_type": "COSINE",  #Ähnlichkeitsmaß für die Suche

    "params": {
        "M": 16,  # Anzahl der Verbindungen pro Knoten
        "efConstruction": 200  # Genauigkeit vs. Geschwindigkeit beim Indexaufbau
    }
}

collection.create_index(
    field_name="embedding",
    index_params=index_params
)

collection.load()

def semantic_search_json(query, k=5):

    q_emb = embedding_model.encode([query]).astype("float32").tolist()

    results = collection.search(
        data=q_emb,
        anns_field="embedding",
        param={
            "metric_type": "COSINE",
            "params": {"ef": 50}
        },
        limit=k,
        output_fields=["text", "name", "ort", "region"]
    )

    output = []

    for hits in results:
        for hit in hits:
            output.append({
                "name": hit.entity.get("name"),
                "ort": hit.entity.get("ort"),
                "region": hit.entity.get("region"),
                "text": hit.entity.get("text"),
                "score": hit.score
            })

    return output



--- Dokument 1: Gedenkstätte und Museum Sachsenhausen ---

Name: Gedenkstätte und Museum Sachsenhausen
            Ort: Oranienburg
            Region: Brandenburg
            Öffnungszeiten: täglich 08:30 - 17:00 Uhr
            Eintrittspreise: Erwachsene: 10 €, Ermäßigt: 5 €, Kinder unter 18 Jahren: frei
            Zielgruppe: Erwachsene, Jugendliche, Kinder
            Kategorie: Museum
            Beschreibung: Die Gedenkstätte und das Museum Sachsenhausen erinnern an die Opfer des Konzentrationslagers Sachsenhausen, das von 1936 bis 1945 in Betrieb war. Besucher können die originalen Lagergebäude, Ausstellungen und Gedenkstätten besichtigen, um mehr über die Geschichte des Ortes und die Schicksale der Häftlinge zu erfahren....

--- Dokument 2: Museum Barberini ---

Name: Museum Barberini
            Ort: Potsdam
            Region: Brandenburg
            Öffnungszeiten: Dienstag bis Sonntag 10:00 - 19:00 Uhr, Dienstag geschlossen
            Eintrittspreise: Erwachsene: 16-18 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]